# Exp125 - training throughput calibration

Diagnostic. **Trains nothing to completion, writes no submission, costs no slot.**

## Why

Every axis we have left is either exhausted on the leaderboard (ILP disappearance
`1.4`-`1.8` flat, division permissiveness `0.902`-`0.903`) or unmeasurable
offline (Exp121 showed the local harness inverts ranking on graph construction).
Meanwhile we are tuning around a model we did not train: `pilkwang`'s
50-epoch `split_0` checkpoint, the only weights in the support pack.

The official repo ships `scripts/train_unet_transformer.py`, and its README says
the released baseline *"was not trained to convergence - expect gains from
training longer"*. With over two months to the deadline, training is the one
direction whose value does not depend on the broken offline ranking: a second
independently-trained model can be ensembled at the probability-map level, and
more capacity is a prior we can hold without local validation.

## What this kernel measures

Before committing multi-hour GPU runs against a limited weekly quota, measure:

1. how long one training iteration takes on a T4 at a batch size that fits;
2. how many train movies the seed-0 split actually yields;
3. projected wall-clock for 10 / 25 / 50 epochs.

It runs a handful of iterations via `--max-iters` and extrapolates. Nothing is
kept.

## Note on splits

With no splits file the script builds a **single** deterministic seed-0 fold
(90/10), so `--split 1` would fail. Training a genuinely different model for
ensembling requires supplying our own splits JSON - scoped here, not done.


In [ ]:
try:
    import zarr, geff, tracksdata
    
except: 
    import os
    import sys
    import subprocess
    import importlib
    import importlib.util
    from pathlib import Path
    
    # Polars wheel in the support package uses the 32-bit runtime package.
    os.environ.setdefault("POLARS_PREFER_PKG", "32")
    
    SUPPORT_DIR = Path(
        "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1"
    )
    WHEELS_DIR = SUPPORT_DIR / "wheels"
    
    if not WHEELS_DIR.exists():
        # Fallback for Kaggle's shorter mounted dataset path.
        candidates = list(Path("/kaggle/input").glob("**/wheels"))
        if not candidates:
            raise FileNotFoundError("Could not find the attached offline wheels directory")
        WHEELS_DIR = candidates[0]
    
    print("Offline wheels:", WHEELS_DIR)
    
    
    # These are packages supplied by the attached dataset.
    # Deliberately exclude:
    #   numpy, scipy, torch, pandas, dask, xarray, numba, llvmlite
    #
    # Kaggle already supplies compatible versions of those packages.
    OFFLINE_PACKAGES = [
        "tracksdata",
        "zarr==3.2.1",
        "numcodecs==0.15.1",
        "donfig==0.8.1.post1",
        "geff==1.2.0.1.1",
        "geff-spec==1.1.1",
        "pyscipopt==6.2.1",
        "ilpy==0.6.0",
        "rustworkx==0.18.0",
        "polars==1.42.0",
        "polars-runtime-32==1.42.0",
        "bidict==0.23.1",
        "imagecodecs==2026.6.26",
    ]
    
    
    def module_missing(module_name: str) -> bool:
        return importlib.util.find_spec(module_name) is None
    
    
    # Import name differs from pip package name in several cases.
    REQUIRED_IMPORTS = {
        "tracksdata": "tracksdata",
        "zarr": "zarr",
        "numcodecs": "numcodecs",
        "geff": "geff",
        "pyscipopt": "pyscipopt",
        "ilpy": "ilpy",
        "rustworkx": "rustworkx",
        "polars": "polars",
        "imagecodecs": "imagecodecs",
    }
    
    
    def purge_modules(module_roots):
        """
        Remove already-imported package modules from sys.modules.
    
        Normally this cell runs before imports, but this also protects against
        accidental imports performed by earlier Kaggle initialization code.
        """
        for root in module_roots:
            for name in list(sys.modules):
                if name == root or name.startswith(root + "."):
                    sys.modules.pop(name, None)
    
    
    def install_offline_packages():
        cmd = [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--no-index",
            "--no-deps",
            "--find-links",
            str(WHEELS_DIR),
            *OFFLINE_PACKAGES,
        ]
    
        print("Installing attached packages without modifying NumPy/SciPy/Torch...")
        result = subprocess.run(
            cmd,
            text=True,
            capture_output=True,
        )
    
        if result.returncode != 0:
            print(result.stdout[-4000:])
            print(result.stderr[-4000:])
            raise RuntimeError("Offline dependency installation failed")
    
        purge_modules(REQUIRED_IMPORTS.values())
    
    
    install_offline_packages()
    
    
    # Verify the complete import chain needed by biohub_tracking.io.
    failures = {}
    
    for name, module_name in {
        **REQUIRED_IMPORTS,
        "numpy": "numpy",
        "scipy": "scipy",
        "dask": "dask.array",
        "xarray": "xarray",
    }.items():
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[name] = f"{type(exc).__name__}: {exc}"
    
    if failures:
        raise ImportError(
            "Dependency verification failed:\n"
            + "\n".join(f"{name}: {error}" for name, error in failures.items())
        )

import zarr, geff, tracksdata
print("All offline dependencies imported successfully.")

In [ ]:
# Locate the official repo (training script + package) and the competition data.
import os, sys, glob, json, time, subprocess, shutil
from pathlib import Path

_m = []
for pat in ("/kaggle/input/*/*/*/src/tracking_cellmot/metrics.py",
            "/kaggle/input/*/*/src/tracking_cellmot/metrics.py",
            "/kaggle/input/**/tracking_cellmot/metrics.py"):
    _m += glob.glob(pat, recursive=True)
assert _m, "official scorer dataset not attached"
SRC = Path(_m[0]).parents[1]                 # .../src
REPO = SRC.parent                            # .../tracking_cellmot_<sha>
TRAIN_SCRIPT = REPO / "scripts" / "train_unet_transformer.py"
assert TRAIN_SCRIPT.exists(), f"missing {TRAIN_SCRIPT}"

COMP = Path("/kaggle/input/competitions/biohub-cell-tracking-during-development")
if not COMP.exists():
    cands = glob.glob("/kaggle/input/*biohub-cell-tracking*")
    assert cands, "competition data not attached"
    COMP = Path(cands[0])
TRAIN_DIR = COMP / "train"
zarrs = sorted(TRAIN_DIR.glob("*.zarr"))
labelled = [p for p in zarrs if (TRAIN_DIR / f"{p.name[:-5]}.geff").exists()]
print(f"repo         : {REPO}")
print(f"train dir    : {TRAIN_DIR}")
print(f"train movies : {len(zarrs)} zarr, {len(labelled)} with .geff labels")
n_val = max(1, len(labelled)//10)
print(f"seed-0 split : {len(labelled)-n_val} train / {n_val} val (single fold 0)")
print(f"GPUs         : {os.popen('nvidia-smi -L').read().strip()}")


In [ ]:
# Short training run to measure iteration throughput. Nothing is retained.
WORK = Path("/kaggle/working/traincal"); WORK.mkdir(exist_ok=True)
MAX_ITERS = int(os.environ.get("CAL_MAX_ITERS", "24"))
BATCH     = int(os.environ.get("CAL_BATCH", "4"))   # T4 has 16GB; default 16 may OOM

env = dict(os.environ)
env["PYTHONPATH"] = str(SRC)
cmd = [sys.executable, str(TRAIN_SCRIPT),
       "--data-dir", str(TRAIN_DIR),
       "--split", "0",
       "--epochs", "1",
       "--max-iters", str(MAX_ITERS),
       "--batch-size", str(BATCH),
       "--num-workers", "2",
       "--single-gpu"]
print("running:", " ".join(cmd), flush=True)
t0 = time.time()
p = subprocess.run(cmd, cwd=str(WORK), env=env, capture_output=True, text=True)
elapsed = time.time() - t0
print(f"\nreturncode={p.returncode}  elapsed={elapsed:.1f}s")
print("--- stdout tail ---"); print(p.stdout[-4000:])
if p.returncode != 0:
    print("--- stderr tail ---"); print(p.stderr[-4000:])


In [ ]:
# Projection
if p.returncode != 0:
    print("Calibration run FAILED - fix before projecting. See stderr above.")
else:
    per_iter = elapsed / MAX_ITERS
    n_train = len(labelled) - n_val
    print(f"batch size          : {BATCH}")
    print(f"iterations measured : {MAX_ITERS}")
    print(f"wall clock          : {elapsed:.1f}s  ({per_iter:.2f}s/iter, includes startup)")
    print()
    print("NOTE: startup (imports, data indexing) is amortised into per_iter here,")
    print("      so these projections are an UPPER bound. Re-measure with more iters")
    print("      before trusting them for a long run.")
    print()
    for iters_per_epoch in (n_train, n_train*2):
        print(f"  assuming {iters_per_epoch} iters/epoch:")
        for ep in (10, 25, 50):
            hrs = per_iter * iters_per_epoch * ep / 3600
            flag = "  << exceeds 9h kernel limit" if hrs > 9 else ""
            print(f"     {ep:>2} epochs -> {hrs:6.2f} h{flag}")


In [ ]:
# Guard: this kernel must not emit a submission.
for _p in ["submission.csv", "/kaggle/working/submission.csv"]:
    assert not os.path.exists(_p), f"unexpected submission at {_p}"
print("confirmed: no submission.csv; costs no submission slot")
